# 01. 定义Jit装饰的算子

In [ ]:
import tilelang
import tilelang.language as T

@tilelang.jit(out_idx=[-1], pass_configs={
    tilelang.PassConfigKey.TL_ASCEND_AUTO_SYNC: True,
})
def my_add():
    @T.prim_func
    def main(
        A: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
        B: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
        C: T.Tensor(shape=(1, 1), dtype="float32"), # type: ignore
    ):
        a_ub = T.alloc_ub(A.shape, A.dtype)
        b_ub = T.alloc_ub(B.shape, B.dtype)
        c_ub = T.alloc_ub(C.shape, C.dtype)
        T.copy(A[0, 0], a_ub)
        T.copy(B[0, 0], b_ub)
        T.tile.add(c_ub, a_ub, b_ub)
        T.copy(c_ub, C[0, 0])
    return main

my_jit_add = my_add()
# print(my_jit_add.get_kernel_source())


# 02. 调用算子&评测

In [ ]:
import torch

torch.manual_seed(0)

a = torch.rand((1, 1), dtype=torch.float32).npu()
b = torch.rand((1, 1), dtype=torch.float32).npu()
c_golden = a + b

print(f"Input tensor a: {a}")
print(f"Input tensor b: {b}")
print(f"Golden tensor c: {c_golden}")

c = my_jit_add(a, b)
print(f"Output tensor c: {c}")

torch.testing.assert_close(c, c_golden, rtol=1e-5, atol=1e-5)
print("Test passed: The output tensor c is equal to the sum of the input tensors a and b.")